<a href="https://colab.research.google.com/github/mintycake420/Basic-Exercises-for-courses/blob/main/InformationRetreival_EX06_211718366.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#InformationRetreival_EX06_211718366.ipynb
# OCR & Information Retrieval with Character Trigrams
## Handling OCR Errors Using Character N-grams

This notebook demonstrates how to handle OCR errors in information retrieval by using character-level trigrams (3-grams) instead of exact word matching.


**Submitted by:** Yotam Katz  
**Date:** 14 December 2025  
**Course:** אחזור מידע 26 3700 א01  
**Lecturer:** Dr. Moshe Friedman  
**ID:** 211718366  
**Email:** Yotamkatz2000@gmail.com

## What the Notebook Contains

1. **Data Collection**
   - Fetches **20 Crusade-related Wikipedia articles**
   - Extracts **one representative sentence** from each article

2. **OCR Noise Simulation**
   Introduces realistic OCR-style errors commonly found in scanned historical texts, such as:
   - **l ↔ I**  
     *crusade → cmsade*  
   - **o ↔ 0**  
     *military → miIitary*  
   - **rn ↔ m**  
     *jerusalem → jemsalem*

3. **Clean Search Query**
   ```text
   "crusade military expedition jerusalem"


## Step 1: Setup and Imports

In [71]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [72]:
# Install required package
!pip install wikipedia-api -q

import wikipediaapi
import numpy as np
from collections import Counter, defaultdict
import re
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Initialize Wikipedia API
wiki = wikipediaapi.Wikipedia(
    language='en',
    user_agent='IR_OCR_Assignment/1.0 (Educational)'
)

## Step 2: Fetch Crusade-Related Articles

We'll get sentences from various Crusade-related Wikipedia articles.

In [73]:
# Crusade-related topics
crusade_topics = [
    'First Crusade',
    'Second Crusade',
    'Third Crusade',
    'Fourth Crusade',
    'Richard the Lionheart',
    'Saladin',
    'Knights Templar',
    'Knights Hospitaller',
    'Siege of Jerusalem (1099)',
    'Battle of Hattin',
    'Pope Urban II',
    'Crusader states',
    'Kingdom of Jerusalem',
    'County of Edessa',
    'Principality of Antioch',
    'Godfrey of Bouillon',
    'Baldwin I of Jerusalem',
    'Siege of Acre (1189–1191)',
    'Battle of Arsuf',
    'Krak des Chevaliers'
]

def get_first_sentence(topic):
    """Get the first meaningful sentence from a Wikipedia article."""
    try:
        page = wiki.page(topic)
        if not page.exists():
            return ""

        # Get summary and extract first sentence
        summary = page.summary
        sentences = re.split(r'(?<=[.!?])\s+', summary)
        return sentences[0] if sentences else ""
    except Exception as e:
        print(f"Error fetching {topic}: {e}")
        return ""

# Collect sentences
print("Fetching Crusade-related articles...\n")
documents = []
for i, topic in enumerate(crusade_topics[:20], 1):
    sentence = get_first_sentence(topic)
    if sentence:
        documents.append(sentence)
        print(f"Doc {i:2d} ({topic}):")
        print(f"  {sentence[:100]}...\n")

print(f"\nCollected {len(documents)} documents.")

Fetching Crusade-related articles...

Doc  1 (First Crusade):
  The First Crusade (1096–1099) was the first of a series of religious wars, or Crusades, which were i...

Doc  2 (Second Crusade):
  The Second Crusade (1147–1149) was the second major crusade launched from Europe....

Doc  3 (Third Crusade):
  The Third Crusade (1189–1192) was an attempt led by King Philip II of France, King Richard I of Engl...

Doc  4 (Fourth Crusade):
  The Fourth Crusade (1202–1204) was a Latin Christian armed expedition called by Pope Innocent III....

Doc  5 (Richard the Lionheart):
  Richard I (8 September 1157 – 6 April 1199), known as Richard the Lionheart or Richard Cœur de Lion ...

Doc  6 (Saladin):
  Salah ad-Din Yusuf ibn Ayyub (c....

Doc  7 (Knights Templar):
  The Poor Fellow-Soldiers of Christ and of the Temple of Solomon, mainly known as the Knights Templar...

Doc  8 (Knights Hospitaller):
  The Order of Knights of the Hospital of Saint John of Jerusalem, commonly known as the Knights H

## Step 3: Introduce OCR Errors


In our project we will introduce the following OCR errors, based on realistic mistakes:


### Document 1 — *First Crusade*
- **"Crusade" → "Cmsade"**  
  - Error type: **rn → m substitution**

### Document 3 — *Third Crusade*
- **"attempt" → "attempl"**  
  - Error type: **t → l substitution**

### Document 4 — *Fourth Crusade*
- **"Christian" → "Chr1stian"**  
  - Error type: **i → 1 substitution**

### Document 9 — *Siege of Jerusalem*
- **"Jerusalem" → "Jemsalem"**  
  - Error type: **ru → m substitution**

### Document 13 — *Kingdom of Jerusalem*
- **"Kingdom" → "K1ngdom"**  
  - Error type: **i → 1 substitution**


In [74]:
# Manual OCR error introduction for guaranteed results
documents_with_errors = documents.copy()

# Manually corrupt specific documents to guarantee OCR errors
manual_corruptions = {
    0: lambda s: s.replace('Crusade', 'Cmsade', 1),           # rn->m error
    2: lambda s: s.replace('attempt', 'attempl', 1),          # t->l error
    3: lambda s: s.replace('Christian', 'Chr1stian', 1),      # i->1 error
    8: lambda s: s.replace('Jerusalem', 'Jemsalem', 1),       # ru->m error
    12: lambda s: s.replace('Kingdom', 'K1ngdom', 1),         # i->1 error
}

for idx, corrupt_func in manual_corruptions.items():
    if idx < len(documents_with_errors):
        documents_with_errors[idx] = corrupt_func(documents_with_errors[idx])

print("Manual OCR Error Introduction:")
print("=" * 80)
print(f"Total documents: {len(documents)}")
print(f"Documents with errors: {len(manual_corruptions)}\n")

for idx in sorted(manual_corruptions.keys()):
    if idx < len(documents):
        print(f"\nDoc {idx+1}:")
        print(f"Original:  {documents[idx][:80]}...")
        print(f"Corrupted: {documents_with_errors[idx][:80]}...")
        if documents[idx] != documents_with_errors[idx]:
            print("  ✓ Error successfully introduced")
        else:
            print("  ✗ No change (word not found)")

Manual OCR Error Introduction:
Total documents: 20
Documents with errors: 5


Doc 1:
Original:  The First Crusade (1096–1099) was the first of a series of religious wars, or Cr...
Corrupted: The First Cmsade (1096–1099) was the first of a series of religious wars, or Cru...
  ✓ Error successfully introduced

Doc 3:
Original:  The Third Crusade (1189–1192) was an attempt led by King Philip II of France, Ki...
Corrupted: The Third Crusade (1189–1192) was an attempl led by King Philip II of France, Ki...
  ✓ Error successfully introduced

Doc 4:
Original:  The Fourth Crusade (1202–1204) was a Latin Christian armed expedition called by ...
Corrupted: The Fourth Crusade (1202–1204) was a Latin Chr1stian armed expedition called by ...
  ✓ Error successfully introduced

Doc 9:
Original:  The siege of Jerusalem marked the successful end of the First Crusade, whose obj...
Corrupted: The siege of Jemsalem marked the successful end of the First Crusade, whose obje...
  ✓ Error successfully introd

## Step 4: Create Query

We'll create a clean query (without errors) containing 3-4 words.

In [75]:
# Clean query (no OCR errors)
query = "crusade military expedition jerusalem"

print(f"Query: {query}")
print(f"Number of terms: {len(query.split())}")

Query: crusade military expedition jerusalem
Number of terms: 4


## Step 5: Generate Character Trigrams (3-grams)

We'll break each word into overlapping 3-character sequences.

For example: `"crusade"` → `["cru", "rus", "usa", "sad", "ade"]`

In [76]:
def get_character_trigrams(text):
    """
    Extract character trigrams from text.

    Args:
        text: Input text string

    Returns:
        List of character trigrams
    """
    # Convert to lowercase and extract words
    words = re.findall(r'\b\w+\b', text.lower())

    trigrams = []
    for word in words:
        # Add word boundaries to help with matching
        padded_word = f"${word}$"  # $ represents word boundary

        # Extract all trigrams
        word_trigrams = [padded_word[i:i+3] for i in range(len(padded_word) - 2)]
        trigrams.extend(word_trigrams)

    return trigrams

# Example: Show trigrams for a sample word
sample_word = "crusade"
sample_trigrams = get_character_trigrams(sample_word)
print(f"Trigrams for '{sample_word}': {sample_trigrams}")

# Show what happens with an OCR error
corrupted_word = "cmsade"  # rn→m error
corrupted_trigrams = get_character_trigrams(corrupted_word)
print(f"Trigrams for '{corrupted_word}': {corrupted_trigrams}")

# Calculate overlap
overlap = len(set(sample_trigrams) & set(corrupted_trigrams))
print(f"\nShared trigrams: {overlap}/{len(sample_trigrams)} ({overlap/len(sample_trigrams)*100:.1f}%)")

Trigrams for 'crusade': ['$cr', 'cru', 'rus', 'usa', 'sad', 'ade', 'de$']
Trigrams for 'cmsade': ['$cm', 'cms', 'msa', 'sad', 'ade', 'de$']

Shared trigrams: 3/7 (42.9%)
This is why trigrams work well for OCR errors!


## Step 6: Build Trigram Index for Documents

In [77]:
# Build trigram index for all documents
print("Building trigram index for documents...\n")

doc_trigram_vectors = []
all_trigrams = set()

# First pass: collect all unique trigrams
for doc in documents_with_errors:
    trigrams = get_character_trigrams(doc)
    all_trigrams.update(trigrams)

# Create a vocabulary (trigram to index mapping)
trigram_vocab = {trigram: idx for idx, trigram in enumerate(sorted(all_trigrams))}
print(f"Total unique trigrams in corpus: {len(trigram_vocab)}")

# Second pass: create trigram frequency vectors for each document
for i, doc in enumerate(documents_with_errors):
    trigrams = get_character_trigrams(doc)
    trigram_counts = Counter(trigrams)

    # Create vector
    vector = np.zeros(len(trigram_vocab))
    for trigram, count in trigram_counts.items():
        vector[trigram_vocab[trigram]] = count

    doc_trigram_vectors.append(vector)

# Convert to numpy array for easier manipulation
doc_trigram_matrix = np.array(doc_trigram_vectors)
print(f"Document-trigram matrix shape: {doc_trigram_matrix.shape}")
print(f"(rows=documents, columns=unique trigrams)")

Building trigram index for documents...

Total unique trigrams in corpus: 813
Document-trigram matrix shape: (20, 813)
(rows=documents, columns=unique trigrams)


## Step 7: Create Query Trigram Vector

In [78]:
# Get query trigrams
query_trigrams = get_character_trigrams(query)
print(f"Query trigrams ({len(query_trigrams)} total):")
print(query_trigrams)

# Create query vector
query_trigram_counts = Counter(query_trigrams)
query_vector = np.zeros(len(trigram_vocab))

for trigram, count in query_trigram_counts.items():
    if trigram in trigram_vocab:
        query_vector[trigram_vocab[trigram]] = count

print(f"\nQuery vector shape: {query_vector.shape}")
print(f"Non-zero elements: {np.count_nonzero(query_vector)}")

Query trigrams (34 total):
['$cr', 'cru', 'rus', 'usa', 'sad', 'ade', 'de$', '$mi', 'mil', 'ili', 'lit', 'ita', 'tar', 'ary', 'ry$', '$ex', 'exp', 'xpe', 'ped', 'edi', 'dit', 'iti', 'tio', 'ion', 'on$', '$je', 'jer', 'eru', 'rus', 'usa', 'sal', 'ale', 'lem', 'em$']

Query vector shape: (813,)
Non-zero elements: 32


## Step 8: Calculate Cosine Similarity

We'll compute the cosine similarity between the query vector and each document vector.



In [79]:
# Calculate cosine similarity between query and all documents
similarities = cosine_similarity(query_vector.reshape(1, -1), doc_trigram_matrix)[0]

print("Cosine Similarities:")
print("=" * 50)
for i, sim in enumerate(similarities, 1):
    print(f"Doc {i:2d}: {sim:.4f}")

Cosine Similarities:
Doc  1: 0.1382
Doc  2: 0.2891
Doc  3: 0.2017
Doc  4: 0.3324
Doc  5: 0.1684
Doc  6: 0.0331
Doc  7: 0.1500
Doc  8: 0.2385
Doc  9: 0.2304
Doc 10: 0.1186
Doc 11: 0.0000
Doc 12: 0.1634
Doc 13: 0.3708
Doc 14: 0.1576
Doc 15: 0.2042
Doc 16: 0.0382
Doc 17: 0.1472
Doc 18: 0.1722
Doc 19: 0.1837
Doc 20: 0.0653


## Step 9: Rank and Present Results

In [80]:
# Create results dataframe
results = []
for i, (sim, doc_clean, doc_ocr) in enumerate(zip(similarities, documents, documents_with_errors), 1):
    has_errors = doc_clean != doc_ocr
    results.append({
        'Doc_ID': i,
        'Similarity': sim,
        'Has_OCR_Errors': has_errors,
        'Document_Preview': doc_ocr[:80] + '...'
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Similarity', ascending=False).reset_index(drop=True)
results_df['Rank'] = range(1, len(results_df) + 1)

# Reorder columns
results_df = results_df[['Rank', 'Doc_ID', 'Similarity', 'Has_OCR_Errors', 'Document_Preview']]

print("\n" + "=" * 100)
print(f"RESULTS FOR QUERY: '{query}'")
print("=" * 100)
print(results_df.to_string(index=False))


RESULTS FOR QUERY: 'crusade military expedition jerusalem'
 Rank  Doc_ID  Similarity  Has_OCR_Errors                                                                    Document_Preview
    1      13    0.370817            True The K1ngdom of Jerusalem, also known as the Crusader Kingdom, was one of the Cru...
    2       4    0.332363            True The Fourth Crusade (1202–1204) was a Latin Chr1stian armed expedition called by ...
    3       2    0.289122           False The Second Crusade (1147–1149) was the second major crusade launched from Europe...
    4       8    0.238513           False The Order of Knights of the Hospital of Saint John of Jerusalem, commonly known ...
    5       9    0.230441            True The siege of Jemsalem marked the successful end of the First Crusade, whose obje...
    6      15    0.204156           False The Principality of Antioch (Latin: Principatus Antiochenus; Norman: Princeté de...
    7       3    0.201725            True The Third Crusad

## Step 10: Detailed Analysis of Top Results

In [81]:
# Show detailed comparison for top 5 results
print("\n" + "=" * 100)
print("DETAILED ANALYSIS OF TOP 5 RESULTS")
print("=" * 100)

top_5_indices = results_df.head(5)['Doc_ID'].values - 1  # Convert to 0-indexed

for rank, doc_idx in enumerate(top_5_indices, 1):
    print(f"\n{'='*100}")
    print(f"RANK {rank} - Document {doc_idx + 1} (Similarity: {similarities[doc_idx]:.4f})")
    print(f"{'='*100}")

    print("\nOriginal text:")
    print(documents[doc_idx])

    print("\nWith OCR errors:")
    print(documents_with_errors[doc_idx])

    if documents[doc_idx] != documents_with_errors[doc_idx]:
        print("\n✓ This document contains OCR errors but was still retrieved!")
    else:
        print("\n- This document has no OCR errors")

    # Show trigram overlap with query
    doc_trigrams_set = set(get_character_trigrams(documents_with_errors[doc_idx]))
    query_trigrams_set = set(query_trigrams)
    overlap = len(doc_trigrams_set & query_trigrams_set)

    print(f"\nTrigram overlap with query: {overlap} shared trigrams")


DETAILED ANALYSIS OF TOP 5 RESULTS

RANK 1 - Document 13 (Similarity: 0.3708)

Original text:
The Kingdom of Jerusalem, also known as the Crusader Kingdom, was one of the Crusader states established in the Levant immediately after the First Crusade.

With OCR errors:
The K1ngdom of Jerusalem, also known as the Crusader Kingdom, was one of the Crusader states established in the Levant immediately after the First Crusade.

✓ This document contains OCR errors but was still retrieved!

Trigram overlap with query: 15 shared trigrams

RANK 2 - Document 4 (Similarity: 0.3324)

Original text:
The Fourth Crusade (1202–1204) was a Latin Christian armed expedition called by Pope Innocent III.

With OCR errors:
The Fourth Crusade (1202–1204) was a Latin Chr1stian armed expedition called by Pope Innocent III.

✓ This document contains OCR errors but was still retrieved!

Trigram overlap with query: 17 shared trigrams

RANK 3 - Document 2 (Similarity: 0.2891)

Original text:
The Second Crusade (114

## Step 11: Demonstrate Robustness to OCR Errors

In [82]:
# Count how many documents with OCR errors appear in top 10
top_10_doc_ids = results_df.head(10)['Doc_ID'].values - 1
top_10_with_errors = sum(1 for idx in top_10_doc_ids
                         if documents[idx] != documents_with_errors[idx])

print("\n" + "=" * 100)
print("ROBUSTNESS ANALYSIS")
print("=" * 100)
print(f"\nQuery: '{query}'")
print(f"\nTotal documents: {len(documents)}")
print(f"Documents with OCR errors: {sum(1 for i in range(len(documents)) if documents[i] != documents_with_errors[i])}")
print(f"\nTop 10 results contain {top_10_with_errors} documents WITH OCR errors")
print(f"\n✓ This demonstrates that character trigrams successfully handle OCR errors!")
print(f"  Even with spelling mistakes, documents are still retrieved based on character overlap.")


ROBUSTNESS ANALYSIS

Query: 'crusade military expedition jerusalem'

Total documents: 20
Documents with OCR errors: 5

Top 10 results contain 4 documents WITH OCR errors

✓ This demonstrates that character trigrams successfully handle OCR errors!
  Even with spelling mistakes, documents are still retrieved based on character overlap.
